# Proximal Policy Optimization (PPO) 🚀

## 🚀 Theoretical Background
Proximal Policy Optimization (PPO) is a state-of-the-art policy gradient method used widely in industry (e.g., OpenAI Five, ChatGPT RLHF).
- **Clipped Objective**: Prevents the policy from changing too drastically in one update, ensuring stability.
- **Actor-Critic**: Uses an 'Actor' to decide actions and a 'Critic' to estimate state values.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

class ActorCritic(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(state_size, 32),
            nn.ReLU(),
            nn.Linear(32, action_size),
            nn.Softmax(dim=-1)
        )
        self.critic = nn.Sequential(
            nn.Linear(state_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.actor(x), self.critic(x)

# PPO update: clipped surrogate objective + critic MSE loss
def ppo_update(model, optimizer, states, actions, log_probs_old, returns, advantages, eps_clip=0.2):
    for _ in range(5): # Multiple epochs
        probs, state_values = model.actor(states), model.critic(states)
        dist = torch.distributions.Categorical(probs)
        log_probs = dist.log_prob(actions)
        
        ratios = torch.exp(log_probs - log_probs_old)
        surr1 = ratios * advantages
        surr2 = torch.clamp(ratios, 1-eps_clip, 1+eps_clip) * advantages
        
        loss = -torch.min(surr1, surr2) + 0.5 * F.mse_loss(state_values.flatten(), returns)
        optimizer.zero_grad()
        loss.mean().backward()
        optimizer.step()

# --- Rollout collection + GAE + training loop ---
# ppo_update was previously defined but never called, and the plot
# below it was two independent random walks with no connection to
# the model above. Actually train PPO on a small GridWorld env and
# plot the real per-episode reward.

class GridWorld:
    def __init__(self):
        self.state = 0
    def reset(self):
        self.state = 0
        return self.state
    def step(self, action):
        row, col = divmod(self.state, 4)
        if action == 0: row = max(0, row - 1)    # Up
        elif action == 1: row = min(3, row + 1) # Down
        elif action == 2: col = max(0, col - 1) # Left
        elif action == 3: col = min(3, col + 1) # Right
        self.state = row * 4 + col
        reward = 1.0 if self.state == 15 else -0.01
        done = self.state == 15
        return self.state, reward, done

def one_hot(state_idx, n=16):
    v = np.zeros(n, dtype=np.float32)
    v[state_idx] = 1.0
    return v

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """Generalized Advantage Estimation."""
    advantages = np.zeros(len(rewards), dtype=np.float32)
    gae = 0.0
    next_value = 0.0
    for t in reversed(range(len(rewards))):
        next_non_terminal = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * next_non_terminal - values[t]
        gae = delta + gamma * lam * next_non_terminal * gae
        advantages[t] = gae
        next_value = values[t]
    returns = advantages + np.array(values, dtype=np.float32)
    return advantages, returns

env = GridWorld()
state_size, action_size = 16, 4
model = ActorCritic(state_size, action_size)
optimizer = optim.Adam(model.parameters(), lr=0.01)

max_steps_per_episode = 50
episode_rewards = []

for update in range(150):
    states, actions, rewards, dones, values, log_probs = [], [], [], [], [], []
    s = one_hot(env.reset())
    total_r = 0.0

    for t in range(max_steps_per_episode):
        state_t = torch.FloatTensor(s)
        with torch.no_grad():
            probs, value = model(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)

        next_state_idx, reward, done = env.step(action.item())
        next_s = one_hot(next_state_idx)

        states.append(s)
        actions.append(action.item())
        rewards.append(reward)
        dones.append(float(done))
        values.append(value.item())
        log_probs.append(log_prob.item())

        s = next_s
        total_r += reward
        if done:
            break

    episode_rewards.append(total_r)

    advantages, returns = compute_gae(rewards, values, dones)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    states_t = torch.FloatTensor(np.array(states))
    actions_t = torch.LongTensor(actions)
    log_probs_old_t = torch.FloatTensor(log_probs)
    returns_t = torch.FloatTensor(returns)
    advantages_t = torch.FloatTensor(advantages)

    ppo_update(model, optimizer, states_t, actions_t, log_probs_old_t, returns_t, advantages_t)

plt.figure(figsize=(8, 5))
plt.plot(episode_rewards)
plt.title('PPO Training Reward on GridWorld')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.show()
